<a href="https://colab.research.google.com/github/DecioVdA/ProveComm/blob/main/RandomTree_Riduzione_della_disponibilit%C3%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import requests
from io import BytesIO
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rd_seed = 42  # fissiamo un random seed per riproducibilità e replicabilità
np.random.seed(rd_seed)
random.seed(rd_seed)

rd_seed = 42

In [2]:
url = "https://github.com/DecioVdA/ProveComm/raw/refs/heads/main/DataSetAddestramento.xlsx"
response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))

In [3]:
fig = px.line(df, x = 'Data', y = 'TransitiFRA', title = 'Analisi Transiti 2024').show()

In [4]:
px.box(df, x = 'TransitiFRA', title = 'Analisi Outliers').show()

In [5]:
df['Data'] = pd.to_datetime(df['Data'])
df['Month'] = df['Data'].dt.month

df['Month'] = df['Month'].astype(str)
df['Scolastiche_VDA'] = df['Scolastiche_VDA'].replace({1: 'Si', 0:'No'})

df = df.drop(['Data'], axis = 1)
df



,GiornoSettimana,Scolastiche_VDA,Scolastiche_FRA,Scolastiche_CH,Festivo_ITA,Festivo_FRA,Festivo_CH,Dispo_TU,Dispo_Fre,TransitiFRA,Month
0,Lunedi,Si,ABC,nullo,FE,FE,FE,1.0,1.0,2274,1
1,Martedi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,3160,1
2,Mercoledi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2667,1
3,Giovedi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2633,1
4,Venerdi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2608,1
...,...,...,...,...,...,...,...,...,...,...,...
361,Venerdi,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2789,12
362,Sabato,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,3095,12
363,Domenica,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2673,12
364,Lunedi,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2663,12


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   GiornoSettimana  366 non-null    object 
 1   Scolastiche_VDA  366 non-null    object 
 2   Scolastiche_FRA  366 non-null    object 
 3   Scolastiche_CH   366 non-null    object 
 4   Festivo_ITA      366 non-null    object 
 5   Festivo_FRA      366 non-null    object 
 6   Festivo_CH       366 non-null    object 
 7   Dispo_TU         366 non-null    float64
 8   Dispo_Fre        366 non-null    float64
 9   TransitiFRA      366 non-null    int64  
 10  Month            366 non-null    object 
dtypes: float64(2), int64(1), object(8)
memory usage: 31.6+ KB


In [7]:
categoriche = ['Scolastiche_VDA', 'Scolastiche_FRA', 'Scolastiche_CH', 'Festivo_ITA', 'Festivo_FRA', 'Festivo_CH', 'GiornoSettimana', 'Month']
numeriche = list(set(df.columns)- set(df[categoriche]) - {'TransitiFRA'})

In [8]:
df.shape[0]

366

In [9]:
df = df[df["Dispo_TU"] > 0].copy()

In [10]:
df.shape[0]

257

In [11]:
X = df.drop(['TransitiFRA'], axis = 1)
y = df['TransitiFRA']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=rd_seed)

In [12]:
encoder = OneHotEncoder().fit(X_train[categoriche])
X_train_cat_enc = encoder.transform(X_train[categoriche]).todense().astype(int)
X_train_cat_enc = pd.DataFrame(X_train_cat_enc, columns = encoder.get_feature_names_out(categoriche), index = X_train.index)
X_test_cat_enc = encoder.transform(X_test[categoriche]).todense().astype(int)
X_test_cat_enc = pd.DataFrame(X_test_cat_enc, columns = encoder.get_feature_names_out(categoriche), index = X_test.index)

X_train = pd.concat([X_train[numeriche], X_train_cat_enc], axis=1)
X_test = pd.concat([X_test[numeriche], X_test_cat_enc], axis=1)
X_train

,Dispo_TU,Dispo_Fre,Scolastiche_VDA_No,Scolastiche_VDA_Si,Scolastiche_FRA_AB,Scolastiche_FRA_ABC,Scolastiche_FRA_AC,Scolastiche_FRA_B,Scolastiche_FRA_C,Scolastiche_FRA_nullo,...,Month_1,Month_12,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9
124,1.0000,1.0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
164,1.0000,1.0,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
245,0.8333,1.0,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
227,1.0000,1.0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
29,1.0000,1.0,0,1,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,1.0000,1.0,1,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
14,1.0000,1.0,1,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
92,1.0000,1.0,1,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
180,1.0000,1.0,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0


In [13]:
model = RandomForestRegressor(random_state = rd_seed)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [14]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [15]:
print("MAE train:", mean_absolute_error(y_train, y_pred_train))
print("MAE test:", mean_absolute_error(y_test, y_pred_test))

print("RMSE train:", mean_squared_error(y_train, y_pred_train))
print("RMSE test:", mean_squared_error(y_test, y_pred_test))

print("R2 train:", r2_score(y_train, y_pred_train))
print("R2 test:", r2_score(y_test, y_pred_test))


MAE train: 194.3676358295646
MAE test: 437.9679328449328
RMSE train: 106819.8673112858
RMSE test: 593391.6093796508
R2 train: 0.86829143375144
R2 test: 0.4205894256876809


In [18]:
imps = pd.DataFrame()
imps['Variabile'] = X_train.columns
imps['Importanza'] = model.feature_importances_
imps = imps.sort_values('Importanza')
px.bar(imps, x = 'Variabile', y = 'Importanza')

In [17]:
media = df['TransitiFRA'].mean()
MAE = mean_absolute_error(y_train, y_pred_train)/media

acc_approssimata = 1-(MAE/media)

print('Accuratezza approssimata: ', acc_approssimata)

Accuratezza approssimata:  0.9999737472791165
